# SfmData

> **Created by Codex.**

Collect `Cal3Bundler` cameras and 3D tracks, and turn them into bundle-adjustment graphs.

Source: [`SfmData.h`](https://github.com/borglab/gtsam/blob/develop/gtsam/sfm/SfmData.h)

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/sfm/doc/SfmData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam
import numpy as np

from gtsam import symbol_shorthand

C = symbol_shorthand.C
K = symbol_shorthand.K
P = symbol_shorthand.P
S = symbol_shorthand.S
X = symbol_shorthand.X

## Dataset model

`SfmData` is the central BAL/Bundler-style container. Camera indices in each track address `cameraList()`. Use `FromBalFile` or `FromBundlerFile` for existing datasets, or populate the container directly.

`generalSfmFactors()` returns only measurement factors. `sfmFactorGraph()` can additionally fix a camera and point to remove the similarity gauge. Passing `None` for either fixed index disables that constraint.

In [ ]:
data = gtsam.SfmData()
calibration = gtsam.Cal3Bundler(500.0, 0.0, 0.0, 0.0, 0.0)
camera = gtsam.PinholeCameraCal3Bundler(gtsam.Pose3(), calibration)
data.addCamera(camera)

track = gtsam.SfmTrack(np.array([0.0, 0.0, 5.0]))
track.addMeasurement(0, np.array([0.0, 0.0]))
data.addTrack(track)

graph = data.sfmFactorGraph()
print("cameras:", data.numberCameras())
print("tracks:", data.numberTracks())
print("graph factors:", graph.size())

## Loading and initialization

For a BAL file, a typical start is:

```python
data = gtsam.SfmData.FromBalFile(filename)
graph = data.sfmFactorGraph()
initial = gtsam.initialCamerasAndPointsEstimate(data)
```

For the fastest current CPU bundle-adjustment path, use the point-batched C++ construction described in [`../sfm.md`](../sfm.md); `SfmData.sfmFactorGraph()` creates one projection factor per observation.